# 21.1 Propositional logic

Truth tables let us replace argument by inspection: if every remaining world makes the conclusion true, the conclusion follows.

Propositional logic begins symbolic AI with whole statements that are true or false. Boolean connectives carve the set of possible worlds, and entailment scans all worlds that satisfy the knowledge base.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random

import matplotlib.pyplot as plt

random.seed(2101)


def implication(left, right):
    return (not left) or right


def worlds(variables):
    names = list(variables)
    for values in itertools.product([False, True], repeat=len(names)):
        yield dict(zip(names, values))


def truth_table_entails(variables, kb, query):
    checked = 0
    surviving = []
    countermodels = []
    rows = []
    for world in worlds(variables):
        checked += 1
        kb_values = [formula(world) for formula in kb]
        kb_ok = all(kb_values)
        query_ok = query(world)
        rows.append({"world": world, "kb": kb_ok, "query": query_ok})
        if kb_ok:
            surviving.append(world)
            if not query_ok:
                countermodels.append(world)
    entailed = len(countermodels) == 0
    return {
        "entailed": entailed,
        "checked": checked,
        "models": surviving,
        "countermodels": countermodels,
        "rows": rows,
    }


def make_prop_ladder():
    return [
        {
            "name": "D1 hand KB",
            "variables": ["P", "Q"],
            "kb": [lambda w: w["P"], lambda w: implication(w["P"], w["Q"])],
            "query": lambda w: w["Q"],
            "expected": True,
        },
        {
            "name": "D2 implication chain",
            "variables": ["P", "Q", "R"],
            "kb": [
                lambda w: w["P"],
                lambda w: implication(w["P"], w["Q"]),
                lambda w: implication(w["Q"], w["R"]),
            ],
            "query": lambda w: w["R"],
            "expected": True,
        },
        {
            "name": "D3 conflict plus grouping trap",
            "variables": ["P", "Q", "R", "Blocked"],
            "kb": [
                lambda w: w["P"],
                lambda w: w["Q"],
                lambda w: implication(w["P"] and w["Q"], w["R"]),
                lambda w: implication(w["Blocked"], not w["R"]),
                lambda w: not w["Blocked"],
            ],
            "query": lambda w: w["R"],
            "expected": True,
        },
        {
            "name": "D4 six-variable policy",
            "variables": ["Badge", "Manager", "Training", "Audit", "Enter", "Escalate"],
            "kb": [
                lambda w: w["Badge"],
                lambda w: w["Manager"],
                lambda w: w["Training"],
                lambda w: implication(w["Badge"] and w["Manager"], w["Enter"]),
                lambda w: implication(w["Enter"] and not w["Audit"], w["Escalate"]),
                lambda w: implication(w["Training"], not w["Audit"]),
            ],
            "query": lambda w: w["Escalate"],
            "expected": True,
        },
        {
            "name": "D5 deep ten-variable policy",
            "variables": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
            "kb": [
                lambda w: w["A"],
                lambda w: implication(w["A"], w["B"]),
                lambda w: implication(w["B"], w["C"]),
                lambda w: implication(w["C"], w["D"]),
                lambda w: implication(w["D"], w["E"]),
                lambda w: implication(w["E"], w["F"]),
                lambda w: implication(w["F"], w["G"]),
                lambda w: implication(w["G"], w["H"]),
                lambda w: implication(w["H"], w["I"]),
                lambda w: implication(w["I"], w["J"]),
            ],
            "query": lambda w: w["J"],
            "expected": True,
        },
    ]


## The concept, built once (D1)

We represent each formula as a Python predicate on a Boolean world. The lesson formula is $P	o Q\equiv 
eg P\lor Q$, so the reusable method must scan every assignment, keep the worlds satisfying $KB$, and reject entailment only if a surviving world makes the query false.

In [ ]:

variables = ["P", "Q"]
kb = [lambda w: w["P"], lambda w: implication(w["P"], w["Q"])]
query = lambda w: w["Q"]

d1_result = truth_table_entails(variables, kb, query)

print("assignments", d1_result["checked"])
print("KB models", len(d1_result["models"]))
print("entailed", d1_result["entailed"])


The exact lesson counts are $2^2=4$ assignments, exactly $1$ model of $KB=\{P,P	o Q\}$, and that one model has $Q=1$. The related compound sentence $(P\land Q)	o R$ has $8$ assignments and $1$ failing row.

In [ ]:

compound_failures = []
for row in worlds(["P", "Q", "R"]):
    ok = implication(row["P"] and row["Q"], row["R"])
    if not ok:
        compound_failures.append(row)

assert d1_result["checked"] == 4
assert len(d1_result["models"]) == 1
assert d1_result["models"][0]["Q"] is True
assert d1_result["entailed"] is True
assert len(compound_failures) == 1


## The dataset ladder

F16 uses inline algorithmic instances. Here the ladder grows from a tiny hand-checkable KB to a ten-variable policy chain with many irrelevant counterfactual worlds.

In [ ]:

prop_ladder = make_prop_ladder()

for rung in prop_ladder:
    result = truth_table_entails(rung["variables"], rung["kb"], rung["query"])
    sample = result["rows"][:2]
    print(rung["name"], "variables", len(rung["variables"]), "assignments", result["checked"], "sample", sample)


## Run the same method across D1-D5

The same truth-table entailment method is used on every rung. Correctness is measured against the intended entailment result, and cost is the number of assignments scanned.

In [ ]:

prop_results = []
for rung in prop_ladder:
    result = truth_table_entails(rung["variables"], rung["kb"], rung["query"])
    correct = result["entailed"] == rung["expected"]
    prop_results.append({
        "name": rung["name"],
        "size": len(rung["variables"]),
        "checked": result["checked"],
        "models": len(result["models"]),
        "correct": correct,
        "result": result,
    })

for row in prop_results:
    print(row["name"], row["size"], row["checked"], row["models"], row["correct"])


## Results visualization

The left panels show surviving proof worlds. The right curve shows why truth tables become search: assignments double with every new proposition.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], prop_results):
    model_text = "\n".join(str(model) for model in row["result"]["models"][:4])
    ax.text(0.05, 0.95, model_text or "no models", va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
xs = [row["size"] for row in prop_results]
ys = [row["checked"] for row in prop_results]
ax.plot(xs, ys, marker="o")
ax.set_xlabel("variables")
ax.set_ylabel("assignments scanned")
ax.set_title("search cost vs size")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: checking only the believed world confuses implication with causation and can miss counterfactual rows. On D5, the believed all-true world is not enough evidence for entailment.

In [ ]:

hard = prop_ladder[-1]
believed_world = {name: True for name in hard["variables"]}
wrong_accepts = all(formula(believed_world) for formula in hard["kb"]) and hard["query"](believed_world)
fixed = truth_table_entails(hard["variables"], hard["kb"], hard["query"])

print("wrong single-world proof", wrong_accepts)
print("fixed scans", fixed["checked"])
print("fixed countermodels", fixed["countermodels"])


## Evaluate it + Practice

- Metric: entailment correctness, assignments scanned, and surviving KB models, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add one extra proposition K to D5 and predict the assignment count before running the cell.

Practice: Remove the fact A from D5 and explain why J is no longer entailed.

Practice: Construct a parenthesis trap where P and Q are true but the intended rule differs from the written rule.